In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
(dataset, info) = tfds.load("oxford_iiit_pet",
                            with_info=True,
                            as_supervised=True)

class_names = info.features['label'].names

In [ ]:
dog_labels = tf.constant(
    [i for i, name in enumerate(class_names) if name[0].islower()], dtype=tf.int64
)

def is_dog(image, label):
    return tf.reduce_any(tf.equal(dog_labels, label))

train_original = dataset['train']
test_original = dataset['test']

dog_class_names = [class_names[i] for i in dog_labels.numpy()]
NUM_CLASSES = len(dog_class_names)

In [ ]:
label_map = {original_idx: new_idx for new_idx, original_idx in enumerate(dog_labels.numpy())}
# Create lookup table for efficient mapping
label_table = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(list(label_map.keys()), dtype=tf.int64),
        values=tf.constant(list(label_map.values()), dtype=tf.int64)
    ),
    default_value=-1
)

valid_labels = tf.constant(list(label_map.keys()), dtype=tf.int64)

def filter_valid(image, label):
    return tf.reduce_any(tf.equal(label, valid_labels))

In [ ]:
# Combine all dog images from both train and test
print("Combining all dog images...")
all_dogs = train_original.filter(filter_valid).concatenate(
    test_original.filter(filter_valid)
)

# Get total count (this will consume the dataset, so we recreate it)
total_count = sum(1 for _ in all_dogs)
print(f"Total dog images: {total_count}")

# Recreate the combined dataset
all_dogs = train_original.filter(filter_valid).concatenate(
    test_original.filter(filter_valid)
)

# Shuffle with a seed for reproducibility
all_dogs = all_dogs.shuffle(buffer_size=5000, seed=42)

# Calculate split sizes (70% train, 15% val, 15% test)
train_size = int(0.7 * total_count)
val_size = int(0.15 * total_count)
# test_size will be the remainder

# Split the data
train_ds_raw = all_dogs.take(train_size)
val_ds_raw = all_dogs.skip(train_size).take(val_size)
test_ds_raw = all_dogs.skip(train_size + val_size)

# Verify counts
train_count = sum(1 for _ in train_ds_raw)
val_count = sum(1 for _ in val_ds_raw)
test_count = sum(1 for _ in test_ds_raw)

print(f"\n=== NEW SPLIT ===")
print(f"Training:   {train_count} images ({train_count/total_count*100:.1f}%)")
print(f"Validation: {val_count} images ({val_count/total_count*100:.1f}%)")
print(f"Test:       {test_count} images ({test_count/total_count*100:.1f}%)")

# Recreate datasets after counting (since counting consumed them)
all_dogs = train_original.filter(filter_valid).concatenate(
    test_original.filter(filter_valid)
).shuffle(buffer_size=5000, seed=42)

train_ds_raw = all_dogs.take(train_size)
val_ds_raw = all_dogs.skip(train_size).take(val_size)
test_ds_raw = all_dogs.skip(train_size + val_size)

In [ ]:
# Collect labels for class weight calculation
print("\nCalculating class weights...")
all_labels = []
for image, label in train_ds_raw:
    all_labels.append(label.numpy())

# Map to new indices
y_train = [label_map[label] for label in all_labels]

# Compute class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights computed:")
for i, weight in enumerate(class_weights[:5]):  # Show first 5
    print(f"  Class {i} ({dog_class_names[i]}): {weight:.3f}")

In [ ]:
# Data augmentation pipeline (only for training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# Training preprocessing WITH augmentation
def preprocess_train(image, label):
    # Resize
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    # Normalize to [0,1]
    image = image / 255.0

    # Apply augmentation
    image = data_augmentation(image, training=True)

    # Remap label
    remapped_label = label_table.lookup(label)
    return image, remapped_label

# Validation/Test preprocessing WITHOUT augmentation
def preprocess_test(image, label):
    # Resize
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    # Normalize to [0,1]
    image = image / 255.0

    # NO augmentation

    # Remap label
    remapped_label = label_table.lookup(label)
    return image, remapped_label

In [ ]:
# Training dataset (with augmentation)
train_ds = (
    train_ds_raw
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)

# Validation dataset (no augmentation)
val_ds = (
    val_ds_raw
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Test dataset (no augmentation)
test_ds = (
    test_ds_raw
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("\nDataset pipelines created:")
print(f"Train: {train_count} samples, {train_count // BATCH_SIZE} steps per epoch")
print(f"Val:   {val_count} samples, {val_count // BATCH_SIZE} validation steps")
print(f"Test:  {test_count} samples")

In [ ]:
plt.figure(figsize=(15, 8))

# Take samples from training set
for i, (image, label) in enumerate(train_ds_raw.take(8)):
    plt.subplot(2, 4, i+1)
    plt.imshow(image)
    plt.title(f"{dog_class_names[label_map[label.numpy()]]}")
    plt.axis('off')
plt.suptitle("Sample Training Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
def create_cnn_model(num_classes):
    model = tf.keras.Sequential([
        # First Convolutional Block
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                               input_shape=(224, 224, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),

        # Second Convolutional Block
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),

        # Third Convolutional Block
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.3),

        # Fourth Convolutional Block
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.3),

        # Fifth Convolutional Block
        tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.3),

        # Global Average Pooling and Dense Layers
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.6),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.6),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    return model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
]

In [ ]:
model = create_cnn_model(NUM_CLASSES)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    steps_per_epoch=train_count // BATCH_SIZE,
    validation_steps=val_count // BATCH_SIZE
)

In [ ]:
model.save("dog_breed_model_tuned.keras")
print("Model saved successfully.")

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title("Loss")
plt.legend()

plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Final Test Accuracy: {test_acc:.4f}")
print(f"Final Test Loss: {test_loss:.4f}")

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=dog_class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap='Blues', annot=False, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
def predict_and_display(index):
    image, label = list(test_ds.unbatch())[index]
    input_img = tf.expand_dims(image, axis=0)

    probs = model.predict(input_img, verbose=0)[0]
    pred_class = np.argmax(probs)

    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.title(f"True: {dog_class_names[label.numpy()]}")
    plt.axis('off')

    plt.subplot(1,2,2)
    # Show top 5 predictions
    top_5_indices = np.argsort(probs)[-5:][::-1]
    top_5_probs = probs[top_5_indices] * 100
    top_5_names = [dog_class_names[i] for i in top_5_indices]

    colors = ['green' if i == pred_class else 'blue' for i in range(5)]
    plt.barh(range(5), top_5_probs, color=colors)
    plt.yticks(range(5), top_5_names)
    plt.xlabel('Probability (%)')
    plt.title(f"Predicted: {dog_class_names[pred_class]}")

    plt.tight_layout()
    plt.show()

# Test prediction
predict_and_display(10)

In [ ]:
predict_and_display(50)

In [ ]:
predict_and_display(90)